# Payroll Surprise and Future S&P 500 Returns

## Objective

This analysis examines whether U.S. nonfarm payroll surprises predict future equity returns.

Specifically, we estimate whether unexpected changes in monthly payroll growth contain information about next-month S&P 500 returns.

---

## Data

- **S&P 500 Index (^GSPC)** — Monthly adjusted close (Yahoo Finance)
- **Nonfarm Payrolls (PAYEMS)** — Monthly level data (FRED)

Sample period: January 2005 – February 2026

---

## Variable Construction

### 1. Equity Returns

Monthly log returns:

\[
r_t = \log(P_t) - \log(P_{t-1})
\]

Using log returns ensures stationarity and avoids spurious regression from price levels.

---

### 2. Payroll Change

Monthly first difference in payroll levels:

\[
\Delta Pay_t = Pay_t - Pay_{t-1}
\]

---

### 3. Payroll Surprise (Model-Based)

Because market consensus data is not used, surprise is defined as the expanding-window one-step-ahead forecast error from an ARIMA model:

\[
Surprise_t = \Delta Pay_t - \mathbb{E}_{t-1}[\Delta Pay_t]
\]

The surprise is standardized:

\[
Surprise^{(z)}_t = \frac{Surprise_t}{\sigma_{36m}}
\]

This makes the coefficient interpretable as the return impact of a +1 standard deviation payroll surprise.

---

## Econometric Specification

We estimate a predictive regression:

\[
r_{t+1} = \alpha + \beta \cdot Surprise_t + \phi_0 r_t + \phi_1 r_{t-1} + \varepsilon_{t+1}
\]

Where:

- \( r_{t+1} \) = next-month S&P log return
- Surprise measured at time \( t \)
- Return lags included to control for short-run dynamics
- Standard errors computed using Newey–West (HAC)

---

## Interpretation of Coefficient

Because surprise is standardized:

- A coefficient of -0.004 implies:
  
  > A +1σ payroll surprise predicts approximately -0.4% next-month S&P return.

---

## Conceptual Framework

A negative coefficient is consistent with the macro-finance "discount rate channel":

- Strong payroll growth → inflation pressure
- Inflation pressure → higher expected interest rates
- Higher discount rates → lower equity valuations

This captures the "good news is bad news" effect in tightening regimes.

---

## Limitations

- Surprise is model-implied, not survey-consensus-based.
- Monthly horizon may dilute immediate market reaction.
- Effects may be regime dependent (e.g., inflationary vs. ZIRP environments).

---

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

# -----------------------
# User params
# -----------------------
START = "2005-01-01"
END   = "2026-02-01"

# Dynamic regression lags (monthly data)
P_RET_LAGS  = 1   # AR lags for equity returns
Q_PAY_LAGS  = 2   # distributed lags for payroll growth
HAC_MAXLAGS = 6   # Newey–West max lags (monthly)

# -----------------------
# Helpers
# -----------------------
def get_fred_series(series_id: str) -> pd.Series:
    """
    Pull a FRED series via public CSV (no API key) and handle varying column names.
    Returns a float series indexed by datetime.
    """
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    df = pd.read_csv(url)

    # Identify the date column robustly
    date_candidates = ["DATE", "date", "observation_date", "observation date", "Date"]
    date_col = next((c for c in date_candidates if c in df.columns), None)

    # If none match, fall back to "first column is date"
    if date_col is None:
        date_col = df.columns[0]

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col)

    # Identify the value column robustly
    if series_id in df.columns:
        val_col = series_id
    else:
        # Fallback: choose the first non-date column
        non_date_cols = [c for c in df.columns if c != date_col]
        if not non_date_cols:
            raise RuntimeError(f"FRED CSV for {series_id} has no value column. Columns={list(df.columns)}")
        val_col = non_date_cols[0]

    s = pd.to_numeric(df[val_col].replace(".", np.nan), errors="coerce")
    s.name = series_id
    return s.dropna()

def make_lags(s: pd.Series, nlags: int, prefix: str) -> pd.DataFrame:
    return pd.concat(
        {f"{prefix}_L{i}": s.shift(i) for i in range(1, nlags + 1)},
        axis=1
    )

def adf_report(x: pd.Series, name: str):
    x = x.dropna()
    stat, pval, usedlag, nobs, crit, _ = adfuller(x, autolag="AIC")
    print(f"\nADF test: {name}")
    print(f"  stat={stat:.4f}, p-value={pval:.4g}, usedlag={usedlag}, nobs={nobs}")
    for k, v in crit.items():
        print(f"  crit({k})={v:.4f}")

# -----------------------
# 1) Download monthly data
# -----------------------
sp = yf.download("^GSPC", start=START, end=END, interval="1mo", auto_adjust=False)["Adj Close"]
sp = sp.dropna().sort_index()
if sp.empty:
    raise RuntimeError("S&P data (^GSPC) came back empty. Check ticker/internet/date range.")

pay = get_fred_series("PAYEMS")
pay = pay.loc[(pay.index >= pd.to_datetime(START)) & (pay.index <= pd.to_datetime(END))]
pay = pay.sort_index()
if pay.empty:
    raise RuntimeError("PAYEMS data came back empty from FRED. Check internet/date range.")

# -----------------------
# 2) Transformations (stationary-ish)
# -----------------------
# S&P: monthly log returns
r_sp = np.log(sp).diff()

# Payrolls: monthly log growth
g_pay = np.log(pay).diff()

data = pd.concat([r_sp, g_pay], axis=1).dropna()
data.columns = ["r_sp", "g_pay"]
if data.empty:
    raise RuntimeError("Merged dataset is empty after transforms. Check overlap of dates/frequency.")

# -----------------------
# 3) Timing alignment to avoid look-ahead
# -----------------------
# Use payroll growth at t to predict S&P return at t+1
data["r_sp_fwd1"] = data["r_sp"].shift(-1)
data = data.dropna()

# -----------------------
# 4) ARDL-style regression:
#    r_{t+1} = a + (AR terms in r_sp at t and lags) + (distributed lags of g_pay at t and lags) + e
# -----------------------
Y = data["r_sp_fwd1"]

X_list = []
X_list.append(pd.Series(1.0, index=data.index, name="const"))

# Equity return dynamics
X_list.append(data["r_sp"].rename("r_sp_L0"))
if P_RET_LAGS > 0:
    X_list.append(make_lags(data["r_sp"], P_RET_LAGS, "r_sp"))

# Payroll growth distributed lags
X_list.append(data["g_pay"].rename("g_pay_L0"))
if Q_PAY_LAGS > 0:
    X_list.append(make_lags(data["g_pay"], Q_PAY_LAGS, "g_pay"))

X = pd.concat(X_list, axis=1).dropna()
Y = Y.loc[X.index]

if X.empty or Y.empty:
    raise RuntimeError("Design matrix became empty after lagging. Reduce lags or expand date range.")

# -----------------------
# 5) Fit with Newey–West HAC SEs
# -----------------------
ols = sm.OLS(Y, X).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_MAXLAGS})

print("\n========================")
print("ARDL-style regression with Newey–West HAC SEs")
print("========================")
print(ols.summary())

# -----------------------
# 6) Diagnostics
# -----------------------
print("\n========================")
print("Diagnostics")
print("========================")

adf_report(data["r_sp"], "S&P monthly log returns (r_sp)")
adf_report(data["g_pay"], "Payroll monthly log growth (g_pay)")

lb = acorr_ljungbox(ols.resid, lags=[6, 12], return_df=True)
print("\nLjung–Box residual autocorrelation test:")
print(lb)

arch_stat, arch_pval, _, _ = het_arch(ols.resid, nlags=12)
print(f"\nARCH test (12 lags): stat={arch_stat:.4f}, p-value={arch_pval:.4g}")

# -----------------------
# 7) Economic interpretation helper
# -----------------------
pay_cols = [c for c in X.columns if c.startswith("g_pay")]
cum_beta = float(ols.params[pay_cols].sum()) if pay_cols else np.nan

print("\n========================")
print("Interpretation helper")
print("========================")
print(f"Cumulative payroll-growth effect (sum of g_pay L0..L{Q_PAY_LAGS}): {cum_beta:.6f}")
print("For a 0.10% payroll log-growth shock (~0.001), multiply cum_beta by 0.001 for approx effect on next-month log return.")

[*********************100%***********************]  1 of 1 completed



ARDL-style regression with Newey–West HAC SEs
                            OLS Regression Results                            
Dep. Variable:              r_sp_fwd1   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                 -0.009
Method:                 Least Squares   F-statistic:                     6.737
Date:                Wed, 11 Feb 2026   Prob (F-statistic):           6.69e-06
Time:                        13:43:09   Log-Likelihood:                 430.35
No. Observations:                 249   AIC:                            -848.7
Df Residuals:                     243   BIC:                            -827.6
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX

# -----------------------
# Params
# -----------------------
START = "2005-01-01"
END   = "2026-02-01"

# Surprise model settings
MIN_TRAIN = 36          # min months before we start producing forecasts
ARIMA_ORDER = (1, 0, 1) # on payroll CHANGE (not log growth)
HAC_MAXLAGS = 6

# Regression settings
RET_LAGS = 1            # include r_t (and one lag) as controls

# -----------------------
# Robust FRED pull (no API key)
# -----------------------
def get_fred_series(series_id: str) -> pd.Series:
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    df = pd.read_csv(url)

    # robust date col detection
    date_candidates = ["DATE", "date", "observation_date", "Date"]
    date_col = next((c for c in date_candidates if c in df.columns), df.columns[0])

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col)

    # robust value col detection
    val_col = series_id if series_id in df.columns else [c for c in df.columns if c != date_col][0]

    s = pd.to_numeric(df[val_col].replace(".", np.nan), errors="coerce").dropna()
    s.name = series_id
    return s

def make_lags(s: pd.Series, nlags: int, prefix: str) -> pd.DataFrame:
    return pd.concat({f"{prefix}_L{i}": s.shift(i) for i in range(1, nlags + 1)}, axis=1)

# -----------------------
# 1) Get monthly S&P and payroll levels
# -----------------------
sp = yf.download("^GSPC", start=START, end=END, interval="1mo", auto_adjust=False)["Adj Close"].dropna()
pay = get_fred_series("PAYEMS")
pay = pay.loc[(pay.index >= pd.to_datetime(START)) & (pay.index <= pd.to_datetime(END))].dropna()

# Align monthly timestamps
df = pd.concat([sp, pay], axis=1).dropna()
df.columns = ["sp", "pay"]

# -----------------------
# 2) Build stationary series
# -----------------------
# Equity: monthly log returns
df["r_sp"] = np.log(df["sp"]).diff()

# Payroll: monthly CHANGE in level (PAYEMS is in thousands of employees)
df["d_pay"] = df["pay"].diff()

df = df.dropna()

# -----------------------
# 3) Construct "surprise" via expanding-window 1-step-ahead forecasts
#    surprise_t = d_pay_t - E_{t-1}[d_pay_t]
# -----------------------
d_pay = df["d_pay"].copy()

forecast = pd.Series(index=d_pay.index, dtype=float)

for t in range(MIN_TRAIN, len(d_pay)):
    train = d_pay.iloc[:t]  # up through t-1 (no look-ahead)
    try:
        m = SARIMAX(train, order=ARIMA_ORDER, trend="c",
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        f = m.get_forecast(steps=1).predicted_mean.iloc[0]
        forecast.iloc[t] = f
    except Exception:
        forecast.iloc[t] = np.nan

df["d_pay_fcst"] = forecast
df["pay_surprise"] = df["d_pay"] - df["d_pay_fcst"]

# standardize surprise (z-score) for easier coefficient interpretation
roll_std = df["pay_surprise"].rolling(36, min_periods=24).std()
df["pay_surprise_z"] = df["pay_surprise"] / roll_std

df = df.dropna(subset=["pay_surprise_z"])

# -----------------------
# 4) Timing: use surprise at t to predict next-month return
# -----------------------
df["r_sp_fwd1"] = df["r_sp"].shift(-1)
df = df.dropna(subset=["r_sp_fwd1"])

# -----------------------
# 5) Regression with HAC SEs
#    r_{t+1} = a + b * surprise_t + controls + e_{t+1}
# -----------------------
Y = df["r_sp_fwd1"]

X_parts = [pd.Series(1.0, index=df.index, name="const"),
           df["pay_surprise_z"].rename("pay_surprise_z")]

# Optional return dynamics controls
X_parts.append(df["r_sp"].rename("r_sp_L0"))
if RET_LAGS > 0:
    X_parts.append(make_lags(df["r_sp"], RET_LAGS, "r_sp"))

X = pd.concat(X_parts, axis=1).dropna()
Y = Y.loc[X.index]

model = sm.OLS(Y, X).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_MAXLAGS})
print(model.summary())

# -----------------------
# 6) Interpretation helper
# -----------------------
b = model.params["pay_surprise_z"]
print("\nInterpretation:")
print(f"Coefficient on payroll surprise (z): {b:.6f}")
print("A +1σ payroll surprise is associated with about b change in NEXT-month log return.")
print("Multiply by 100 for approx percent return impact.")

[*********************100%***********************]  1 of 1 completed
c:\Users\Jonbf5\AppData\Local\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Jonbf5\AppData\Local\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Jonbf5\AppData\Local\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Jonbf5\AppData\Local\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Jonbf5\AppData\Local\anaconda3\Lib\sit

                            OLS Regression Results                            
Dep. Variable:              r_sp_fwd1   R-squared:                       0.042
Model:                            OLS   Adj. R-squared:                  0.027
Method:                 Least Squares   F-statistic:                     5.139
Date:                Wed, 11 Feb 2026   Prob (F-statistic):            0.00195
Time:                        13:43:51   Log-Likelihood:                 341.94
No. Observations:                 191   AIC:                            -675.9
Df Residuals:                     187   BIC:                            -662.9
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0120      0.003      3.

c:\Users\Jonbf5\AppData\Local\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Jonbf5\AppData\Local\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
